# 🧠 MBTI Personality Detection — RAG + Multi-Agent + Baselines

**System Overview:**
- **Task**: 4-axis binary MBTI classification (I/E, S/N, T/F, J/P)
- **Models**: SVM+TF-IDF · RoBERTa-mean · D-DGCN · RAG+Multi-Agent
- **Hardware**: Auto-detects 1 GPU (Colab) or 2 GPU (Kaggle T4×2)
- **RAG**: Dual-embedding FAISS (raw text + psycho summary)
- **Agents**: 4 Analysis Agents (parallel) + 1 Synthesis Agent

---
**Datasets supported:**
- MBTI-500 (~106K rows): `/kaggle/input/.../MBTI 500.csv`
- Kaggle MBTI (~8600 rows): `/kaggle/input/.../mbti_1.csv`

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers faiss-cpu scikit-learn torch torchvision torchaudio
!pip install -q torch_geometric sentence-transformers accelerate tqdm openai

## 📥 Cell 2 — Imports

In [ ]:
import os, re, json, time, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import faiss

try:
    from torch_geometric.nn import GCNConv, global_mean_pool
    from torch_geometric.data import Data, Batch
    HAS_PYG = True
except ImportError:
    HAS_PYG = False
    print('⚠️  torch_geometric not available — D-DGCN will be skipped')

warnings.filterwarnings('ignore')
print('✅ Imports done')

## ⚙️ Cell 3 — Config

In [ ]:
# ───────────────────────────────────────────────
# CONFIG — all paths and hyperparameters here
# ───────────────────────────────────────────────
CONFIG = {
    # Paths
    'DATA_PATH_500':  '/kaggle/input/datasets/zeyadkhalid/mbti-personality-types-500-dataset/MBTI 500.csv',
    'DATA_PATH_8K':   '/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv',
    'RESULT_DIR':     '/kaggle/working/results',

    # Data preprocessing
    'MAX_POSTS':      50,
    'MAX_WORDS':      70,

    # Model
    'ROBERTA_NAME':   'roberta-base',
    'MAX_TOK_LEN':    128,
    'BATCH_SIZE':     8,
    'ENCODER_LR':     1e-5,
    'OTHER_LR':       1e-3,
    'MAX_EPOCHS':     10,
    'PATIENCE':       3,

    # GCN
    'GCN_HIDDEN':     128,
    'GCN_OUT':        64,
    'TOP_K_GRAPH':    10,

    # RAG
    'RAG_TOP_K':      5,
    'EMBED_DIM':      768,

    # LLM
    'LLM_PROVIDER':   'openai',   # 'openai' | 'deepseek' | 'local'
    'LLM_MODEL':      'gpt-4o-mini',
    'OPENAI_API_KEY': os.environ.get('OPENAI_API_KEY', ''),
    'DEEPSEEK_KEY':   os.environ.get('DEEPSEEK_API_KEY', ''),
}

os.makedirs(CONFIG['RESULT_DIR'], exist_ok=True)

# ── GPU detection ──
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if NUM_GPUS > 0 else 'cpu')
print(f'🖥  GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')
if NUM_GPUS == 2:
    print('🚀 Kaggle T4×2 detected → will use DataParallel')
elif NUM_GPUS == 1:
    print('🚀 Single GPU detected → standard training')
else:
    print('⚠️  No GPU found → CPU mode')

# ── Constants ──
MBTI_TYPES  = ['infj','infp','intj','intp','isfj','isfp','istj','istp',
               'enfj','enfp','entj','entp','esfj','esfp','estj','estp']
MBTI_EXTRA  = ['introvert','extrovert','introverted','extroverted',
               'sensing','intuition','thinking','feeling','judging','perceiving']
ALL_MASK    = MBTI_TYPES + MBTI_EXTRA
MASK_RE     = re.compile(r'\b(' + '|'.join(ALL_MASK) + r')\b', re.IGNORECASE)

LABEL_COL   = 'type'
TEXT_COL    = 'posts'
LABEL_COLS  = ['label_ie', 'label_sn', 'label_tf', 'label_jp']
DIM_NAMES   = ['I/E', 'S/N', 'T/F', 'J/P']
print('✅ Config ready')

## 📂 Cell 4 — Load & Preprocess Dataset

In [ ]:
# ── Load dataset (prefer 500K, fall back to 8K) ──
def load_dataset(cfg):
    for path in [cfg['DATA_PATH_500'], cfg['DATA_PATH_8K']]:
        try:
            df = pd.read_csv(path)
            print(f'✅ Loaded: {path}  shape={df.shape}')
            return df
        except FileNotFoundError:
            continue
    raise FileNotFoundError('❌ No dataset found. Add data and check paths.')

df_raw = load_dataset(CONFIG)


# ── Step 1: Map 16-class MBTI → 4 binary labels ──
def map_mbti_to_binary(mbti_str):
    m = mbti_str.strip().upper()
    return int(m[0]=='I'), int(m[1]=='N'), int(m[2]=='F'), int(m[3]=='J')

df_raw[['label_ie','label_sn','label_tf','label_jp']] = (
    df_raw[LABEL_COL].apply(lambda x: pd.Series(map_mbti_to_binary(x)))
)


# ── Step 2: Psycholinguistic masking + truncation ──
def preprocess_user_posts(raw_str, max_posts=CONFIG['MAX_POSTS'], max_words=CONFIG['MAX_WORDS']):
    """Split posts, mask MBTI keywords, truncate to max_words each."""
    posts = raw_str.split('|||')
    result = []
    for p in posts[:max_posts]:
        p = MASK_RE.sub('<type>', p.strip())
        p = ' '.join(p.split()[:max_words])
        if p:
            result.append(p)
    return result

print('⏳ Preprocessing posts (masking + truncation)...')
df_raw['processed_posts'] = df_raw[TEXT_COL].apply(preprocess_user_posts)

# Concatenated version for SVM / RAG text input
df_raw['concat_posts'] = df_raw['processed_posts'].apply(lambda x: ' ||| '.join(x))
print(f'✅ Preprocessing done. Rows: {len(df_raw)}')


# ── Step 3: Stratified split across ALL 4 axes jointly ──
df_raw['combo_label'] = df_raw[LABEL_COLS].astype(str).agg(''.join, axis=1)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.4, stratify=df_raw['combo_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['combo_label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\n📊 Split → Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')


# ── Step 4: Class weights from train set ──
class_weights_list = []
for col in LABEL_COLS:
    y = train_df[col].values
    cw = compute_class_weight('balanced', classes=np.unique(y), y=y)
    class_weights_list.append(torch.tensor(cw, dtype=torch.float))
print('✅ Class weights computed:', [cw.tolist() for cw in class_weights_list])

## 🤖 Cell 5 — LLM Client (Pluggable)

In [ ]:
class LLMClient:
    """
    Pluggable LLM client.
    Supports: 'openai', 'deepseek', 'local' (stub)
    """
    def __init__(self, cfg=CONFIG):
        self.provider = cfg['LLM_PROVIDER']
        self.model    = cfg['LLM_MODEL']
        self.cfg      = cfg
        self._client  = None
        self._init_client()

    def _init_client(self):
        if self.provider == 'openai':
            try:
                from openai import OpenAI
                self._client = OpenAI(api_key=self.cfg['OPENAI_API_KEY'])
            except Exception as e:
                print(f'⚠️  OpenAI init failed: {e}. Using stub.')
        elif self.provider == 'deepseek':
            try:
                from openai import OpenAI
                self._client = OpenAI(
                    api_key=self.cfg['DEEPSEEK_KEY'],
                    base_url='https://api.deepseek.com/v1'
                )
                self.model = 'deepseek-chat'
            except Exception as e:
                print(f'⚠️  DeepSeek init failed: {e}. Using stub.')

    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        """Generate text from prompt. Falls back to stub if no API key."""
        if self._client is None or not self.cfg.get('OPENAI_API_KEY') and not self.cfg.get('DEEPSEEK_KEY'):
            return self._stub_response(prompt)
        try:
            resp = self._client.chat.completions.create(
                model=self.model,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=max_tokens,
                temperature=0.3,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'⚠️  LLM error: {e}')
            return self._stub_response(prompt)

    def _stub_response(self, prompt: str) -> str:
        """Deterministic stub for offline/no-key testing."""
        if 'JSON' in prompt or 'json' in prompt:
            return json.dumps({'I/E': 0, 'S/N': 1, 'T/F': 0, 'J/P': 1,
                               'reasoning': 'stub response'})
        return 'This user shows introverted and analytical tendencies based on the posts.'

# Instantiate global LLM client
llm_client = LLMClient(CONFIG)
print('✅ LLM client ready:', llm_client.provider)

## 🔍 Cell 6 — Feature Extraction Agent (Psychological Summary)

In [ ]:
FEATURE_AGENT_PROMPT = """\
You are a psycholinguistic analyst. Given the following social media posts from one person,
generate a concise psychological summary (3-5 sentences) covering:
1. Social orientation (introvert vs extrovert)
2. Information processing style (sensing vs intuition)
3. Decision-making approach (thinking vs feeling)
4. Lifestyle preference (judging vs perceiving)

Posts:
{posts}

Output ONLY the summary text."""


def feature_agent_generate(posts_text: str, llm: LLMClient) -> str:
    """Generate psychological summary for one user's posts."""
    prompt = FEATURE_AGENT_PROMPT.format(posts=posts_text[:1500])  # cap tokens
    return llm.generate(prompt, max_tokens=256)


def extract_features_batch(df_subset, llm: LLMClient, max_workers=4, sample_n=None):
    """
    Run feature agent over a DataFrame subset in parallel.
    sample_n: if set, only process first N rows (for fast testing)
    """
    if sample_n:
        df_subset = df_subset.iloc[:sample_n].copy()

    summaries = [None] * len(df_subset)
    indices   = list(range(len(df_subset)))

    def _worker(i):
        row  = df_subset.iloc[i]
        text = row['concat_posts']
        return i, feature_agent_generate(text, llm)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_worker, i): i for i in indices}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Feature Agent'):
            i, summary = fut.result()
            summaries[i] = summary

    df_subset = df_subset.copy()
    df_subset['psych_summary'] = summaries
    return df_subset


# ⚠️ For large datasets run on a sample to save cost/time.
# Set FEATURE_SAMPLE to None to run on full train set.
FEATURE_SAMPLE = 200  # change to None for full run

print('⏳ Running Feature Extraction Agent on train set...')
train_df_feat = extract_features_batch(train_df, llm_client, max_workers=4, sample_n=FEATURE_SAMPLE)
# For rows not processed, use concat_posts as fallback summary
if FEATURE_SAMPLE and FEATURE_SAMPLE < len(train_df):
    remaining = train_df.iloc[FEATURE_SAMPLE:].copy()
    remaining['psych_summary'] = remaining['concat_posts'].str[:300]
    train_df = pd.concat([train_df_feat, remaining], ignore_index=True)
else:
    train_df = train_df_feat

print(f'✅ Psychological summaries generated. Sample:\n{train_df["psych_summary"].iloc[0][:200]}')

## 📚 Cell 7 — Label Knowledge Base (Label Enrichment)

In [ ]:
LABEL_KB_PROMPTS = {
    'I/E': 'Describe the key linguistic and behavioral markers that distinguish Introverts (I) from Extroverts (E) in online social media posts. Be concise (3 sentences).',
    'S/N': 'Describe the key linguistic markers distinguishing Sensing (S) from Intuition (N) types in social media posts. Be concise (3 sentences).',
    'T/F': 'Describe the key linguistic markers distinguishing Thinking (T) from Feeling (F) types in social media posts. Be concise (3 sentences).',
    'J/P': 'Describe the key linguistic markers distinguishing Judging (J) from Perceiving (P) types in social media posts. Be concise (3 sentences).',
}

print('⏳ Generating label knowledge base...')
label_knowledge_base = {}
for axis, prompt in LABEL_KB_PROMPTS.items():
    label_knowledge_base[axis] = llm_client.generate(prompt, max_tokens=150)
    print(f'  {axis}: {label_knowledge_base[axis][:80]}...')

print('✅ Label knowledge base ready')

## 🗄️ Cell 8 — Build FAISS Index (Dual-RAG)

In [ ]:
from sentence_transformers import SentenceTransformer

# ── Load sentence encoder ──
print('⏳ Loading sentence encoder...')
sent_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
EMBED_DIM    = sent_encoder.get_sentence_embedding_dimension()
print(f'✅ Sentence encoder ready. Embedding dim: {EMBED_DIM}')


def encode_texts(texts, encoder, batch_size=64, desc='Encoding'):
    """Batch-encode texts with progress bar."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        embs  = encoder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        all_embs.append(embs)
    return np.vstack(all_embs).astype('float32')


# ── Dual embedding: raw text + psycho summary ──
print('⏳ Encoding raw texts...')
raw_texts      = train_df['concat_posts'].tolist()
summary_texts  = train_df['psych_summary'].tolist()

raw_embs     = encode_texts(raw_texts,     sent_encoder, desc='Raw text emb')
summary_embs = encode_texts(summary_texts, sent_encoder, desc='Summary emb')

# ── Fuse embeddings: concatenate raw + summary → normalize ──
dual_embs = np.hstack([raw_embs, summary_embs]).astype('float32')  # [N, 2*EMBED_DIM]
faiss.normalize_L2(dual_embs)
DUAL_DIM = dual_embs.shape[1]
print(f'  Dual embedding dim: {DUAL_DIM}')

# ── Build FAISS index ──
faiss_index = faiss.IndexFlatIP(DUAL_DIM)  # Inner product on normalized vecs = cosine
faiss_index.add(dual_embs)
print(f'✅ FAISS index built. Vectors: {faiss_index.ntotal}')

# ── Store metadata for retrieval ──
rag_metadata = train_df[LABEL_COLS + ['concat_posts', 'psych_summary']].copy()
rag_metadata = rag_metadata.reset_index(drop=True)
print('✅ RAG metadata stored')

## 🔎 Cell 9 — Balanced Few-Shot Retriever (Dual-RAG)

In [ ]:
class DualRAGRetriever:
    """
    Retrieves Top-K examples from FAISS index using dual embeddings.
    Applies balanced few-shot selection to include minority labels.
    """
    def __init__(self, index, metadata_df, encoder, embed_dim, class_weights,
                 top_k=5):
        self.index      = index
        self.meta       = metadata_df
        self.encoder    = encoder
        self.embed_dim  = embed_dim  # dim of EACH half (raw or summary)
        self.cw         = class_weights
        self.top_k      = top_k

    def _embed_query(self, raw_text: str, summary: str) -> np.ndarray:
        raw_emb = self.encoder.encode([raw_text], convert_to_numpy=True).astype('float32')
        sum_emb = self.encoder.encode([summary],  convert_to_numpy=True).astype('float32')
        q = np.hstack([raw_emb, sum_emb]).astype('float32')
        faiss.normalize_L2(q)
        return q

    def retrieve(self, raw_text: str, summary: str):
        """
        Balanced retrieval:
        1. Get top 3*K candidates via cosine similarity
        2. Re-rank to ensure minority label coverage
        Returns list of metadata dicts.
        """
        q = self._embed_query(raw_text, summary)
        k_fetch = min(self.top_k * 3, self.index.ntotal)
        _, idxs = self.index.search(q, k_fetch)  # [1, k_fetch]
        idxs     = idxs[0]

        candidates = self.meta.iloc[idxs].copy()
        candidates['_score'] = range(len(candidates))  # rank proxy

        # Balanced selection: for each axis, ensure at least 1 minority sample
        selected_idxs = set()
        for col_idx, col in enumerate(LABEL_COLS):
            cw = self.cw[col_idx]
            # minority class = class with higher weight
            minority_cls = int(cw.argmax().item())
            minority_rows = candidates[candidates[col] == minority_cls]
            if len(minority_rows) > 0:
                selected_idxs.add(minority_rows.index[0])

        # Fill remaining slots with top-ranked
        for idx in candidates.index:
            if len(selected_idxs) >= self.top_k:
                break
            selected_idxs.add(idx)

        result = self.meta.loc[list(selected_idxs)].head(self.top_k)
        return result.to_dict(orient='records')


retriever = DualRAGRetriever(
    index      = faiss_index,
    metadata_df= rag_metadata,
    encoder    = sent_encoder,
    embed_dim  = EMBED_DIM,
    class_weights = class_weights_list,
    top_k      = CONFIG['RAG_TOP_K']
)
print('✅ DualRAGRetriever ready')

## 🤝 Cell 10 — Multi-Agent System (4 Parallel Analysis Agents + Synthesis)

In [ ]:
# ─────────────────────────────────────────────────────────────
# ANALYSIS AGENT — one per MBTI axis
# ─────────────────────────────────────────────────────────────

ANALYSIS_PROMPT = """\
You are a psycholinguist expert in the {axis} MBTI axis.

Axis knowledge: {axis_knowledge}

Analyze the following user posts and predict whether this person leans toward
{label_0} (0) or {label_1} (1) on the {axis} axis.

--- User Posts ---
{posts}

--- Similar Examples (Few-Shot) ---
{examples}

Output a JSON object with exactly two keys:
{{"prediction": <0 or 1>, "confidence": <0.0 to 1.0>}}
Output ONLY valid JSON, nothing else."""

AXIS_LABELS = {
    'I/E': ('Introvert (I)', 'Extrovert (E)'),
    'S/N': ('Sensing (S)',   'Intuition (N)'),
    'T/F': ('Thinking (T)', 'Feeling (F)'),
    'J/P': ('Judging (J)',  'Perceiving (P)'),
}


def analysis_agent(axis: str, posts_text: str, examples: list,
                   kb: dict, llm: LLMClient) -> dict:
    """Single axis analysis agent."""
    ex_str = '\n'.join([
        f"  Ex{i+1}: posts_snippet='{e['concat_posts'][:120]}...' "
        f"→ {axis}={e[LABEL_COLS[DIM_NAMES.index(axis)]]}"
        for i, e in enumerate(examples)
    ])
    prompt = ANALYSIS_PROMPT.format(
        axis         = axis,
        axis_knowledge = kb.get(axis, ''),
        label_0      = AXIS_LABELS[axis][0],
        label_1      = AXIS_LABELS[axis][1],
        posts        = posts_text[:800],
        examples     = ex_str
    )
    raw = llm.generate(prompt, max_tokens=64)
    try:
        result = json.loads(raw)
        return {'axis': axis, 'prediction': int(result.get('prediction', 0)),
                'confidence': float(result.get('confidence', 0.5))}
    except Exception:
        return {'axis': axis, 'prediction': 0, 'confidence': 0.5}


# ─────────────────────────────────────────────────────────────
# SYNTHESIS AGENT
# ─────────────────────────────────────────────────────────────

SYNTH_PROMPT = """\
You are an MBTI expert. Based on the axis predictions below, provide the final
4-axis MBTI classification.

Axis predictions:
{predictions}

Output a JSON object with keys: I/E, S/N, T/F, J/P (each 0 or 1) and a 'reasoning' key.
Example: {{"I/E": 1, "S/N": 0, "T/F": 1, "J/P": 0, "reasoning": "..."}}
Output ONLY valid JSON."""


def synthesis_agent(axis_results: list, llm: LLMClient) -> dict:
    pred_str = '\n'.join([f"  {r['axis']}: prediction={r['prediction']}, confidence={r['confidence']:.2f}"
                          for r in axis_results])
    raw = llm.generate(SYNTH_PROMPT.format(predictions=pred_str), max_tokens=128)
    try:
        result = json.loads(raw)
        return {k: int(result.get(k, 0)) for k in DIM_NAMES}
    except Exception:
        return {k: r['prediction'] for k, r in zip(DIM_NAMES, axis_results)}


# ─────────────────────────────────────────────────────────────
# FULL PIPELINE: RAG → 4 Parallel Agents → Synthesis
# ─────────────────────────────────────────────────────────────

def rag_multi_agent_predict_single(posts_text: str, summary: str,
                                    retriever, llm, kb) -> list:
    """
    Full RAG + multi-agent pipeline for one user.
    Returns: [pred_ie, pred_sn, pred_tf, pred_jp]
    """
    # Step 1: Retrieve similar examples
    examples = retriever.retrieve(posts_text, summary)

    # Step 2: Run 4 Analysis Agents IN PARALLEL
    axis_results = [None] * 4
    with ThreadPoolExecutor(max_workers=4) as executor:
        future_map = {
            executor.submit(analysis_agent, axis, posts_text, examples, kb, llm): i
            for i, axis in enumerate(DIM_NAMES)
        }
        for fut in as_completed(future_map):
            i = future_map[fut]
            axis_results[i] = fut.result()

    # Step 3: Synthesis Agent
    final = synthesis_agent(axis_results, llm)
    return [final[dim] for dim in DIM_NAMES]


def rag_multi_agent_predict_batch(df_subset, retriever, llm, kb, max_workers=2):
    """Run RAG+Multi-Agent on a DataFrame. Returns np.ndarray [N, 4]."""
    n      = len(df_subset)
    preds  = np.zeros((n, 4), dtype=int)

    def _worker(i):
        row     = df_subset.iloc[i]
        summary = row.get('psych_summary', row['concat_posts'][:300])
        pred    = rag_multi_agent_predict_single(
            row['concat_posts'], summary, retriever, llm, kb
        )
        return i, pred

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_worker, i): i for i in range(n)}
        for fut in tqdm(as_completed(futures), total=n, desc='RAG+MultiAgent'):
            i, pred = fut.result()
            preds[i] = pred

    return preds


print('✅ Multi-Agent system defined')

## 🚀 Cell 11 — Run RAG + Multi-Agent on Test Set

In [ ]:
# ⚠️ Set RAG_TEST_SAMPLE to None to evaluate on full test set
RAG_TEST_SAMPLE = 100

test_rag_df = test_df.iloc[:RAG_TEST_SAMPLE].copy() if RAG_TEST_SAMPLE else test_df.copy()

# Generate summaries for test set (or use concat_posts as fallback)
if 'psych_summary' not in test_rag_df.columns:
    test_rag_df['psych_summary'] = test_rag_df['concat_posts'].str[:300]

print(f'⏳ Running RAG+Multi-Agent on {len(test_rag_df)} test samples...')
rag_preds = rag_multi_agent_predict_batch(
    test_rag_df, retriever, llm_client, label_knowledge_base, max_workers=2
)

rag_true = test_rag_df[LABEL_COLS].values

# ── Save RAG predictions ──
rag_save_df = pd.DataFrame(rag_true, columns=[f'y_true_{c}' for c in LABEL_COLS])
for i, c in enumerate(LABEL_COLS):
    rag_save_df[f'y_pred_{c}'] = rag_preds[:, i]
rag_save_df.to_csv(f"{CONFIG['RESULT_DIR']}/rag_multi_agent_predictions.csv", index=False)

rag_scores = {}
for i, name in enumerate(DIM_NAMES):
    rag_scores[name] = f1_score(rag_true[:, i], rag_preds[:, i], average='macro', zero_division=0)
rag_scores['Average'] = np.mean(list(rag_scores.values()))
print(f'✅ RAG+MultiAgent scores: {rag_scores}')
print(f'   Saved → {CONFIG["RESULT_DIR"]}/rag_multi_agent_predictions.csv')

## 📈 Cell 12 — Baseline 1: SVM + TF-IDF

In [ ]:
print('⏳ [SVM] Training...')

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
svm_model  = MultiOutputClassifier(
    SVC(kernel='linear', class_weight='balanced', probability=False, C=1.0),
    n_jobs=-1
)

X_train_text = train_df['concat_posts'].tolist()
X_test_text  = test_df['concat_posts'].tolist()
y_train_svm  = train_df[LABEL_COLS].values
y_test_svm   = test_df[LABEL_COLS].values

# 1. Fit và Transform
X_train_feat = vectorizer.fit_transform(X_train_text)
X_test_feat  = vectorizer.transform(X_test_text)

# 2. THÊM 2 DÒNG NÀY ĐỂ FIX LỖI "READ-ONLY"
X_train_feat.sort_indices()
X_test_feat.sort_indices()

# 3. Fit mô hình
svm_model.fit(X_train_feat, y_train_svm)
svm_preds = svm_model.predict(X_test_feat)

# ── Save SVM predictions ──
svm_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    svm_save[f'y_true_{c}'] = y_test_svm[:, i]
    svm_save[f'y_pred_{c}'] = svm_preds[:, i]
svm_save.to_csv(f"{CONFIG['RESULT_DIR']}/svm_predictions.csv", index=False)

svm_scores = {}
for i, name in enumerate(DIM_NAMES):
    svm_scores[name] = f1_score(y_test_svm[:, i], svm_preds[:, i], average='macro', zero_division=0)
svm_scores['Average'] = np.mean(list(svm_scores.values()))
print(f'✅ SVM scores: {svm_scores}')
print(f'   Saved → {CONFIG["RESULT_DIR"]}/svm_predictions.csv')

## 🗂️ Cell 13 — Dataset & DataLoader for Neural Models

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['ROBERTA_NAME'])


class MBTIDataset(Dataset):
    """
    Dataset returning tokenized posts tensor of shape [MAX_POSTS, MAX_TOK_LEN].
    """
    def __init__(self, dataframe, tok, max_posts=CONFIG['MAX_POSTS'],
                 max_length=CONFIG['MAX_TOK_LEN']):
        self.df         = dataframe.reset_index(drop=True)
        self.tok        = tok
        self.max_posts  = max_posts
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        posts = row['processed_posts'][:self.max_posts]

        enc = self.tok(
            posts, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        ids   = enc['input_ids']       # [P, L]
        masks = enc['attention_mask']  # [P, L]

        P = ids.shape[0]
        if P < self.max_posts:
            pad_ids   = torch.zeros(self.max_posts - P, self.max_length, dtype=torch.long)
            pad_masks = torch.zeros(self.max_posts - P, self.max_length, dtype=torch.long)
            ids   = torch.cat([ids,   pad_ids],   dim=0)
            masks = torch.cat([masks, pad_masks], dim=0)

        labels = torch.tensor([row[c] for c in LABEL_COLS], dtype=torch.float)
        return {'input_ids': ids, 'attention_mask': masks,
                'labels': labels, 'num_posts': P}


train_ds = MBTIDataset(train_df, tokenizer)
val_ds   = MBTIDataset(val_df,   tokenizer)
test_ds  = MBTIDataset(test_df,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ DataLoaders ready — train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}')

## 🤗 Cell 14 — Baseline 2: RoBERTa-mean (Multi-GPU)

In [ ]:
class RoBERTaMeanModel(nn.Module):
    """
    Tensor flow:
    [B,P=50,L] → reshape [B*P,L] → RoBERTa CLS → [B*P,768]
    → reshape [B,P,768] → mean → [B,768] → 4 heads → [B,4]
    """
    def __init__(self, model_name=CONFIG['ROBERTA_NAME'], num_dims=4, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        # Gradient checkpointing to save memory
        self.encoder.gradient_checkpointing_enable()
        h = self.encoder.config.hidden_size
        self.drop  = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(h, 1) for _ in range(num_dims)])

    def forward(self, input_ids, attention_mask, **kwargs):
        B, P, L = input_ids.shape
        ids   = input_ids.view(B*P, L)
        masks = attention_mask.view(B*P, L)
        out   = self.encoder(input_ids=ids, attention_mask=masks)
        cls   = out.last_hidden_state[:, 0, :]     # [B*P, 768]
        user  = cls.view(B, P, -1).mean(dim=1)    # [B, 768]
        user  = self.drop(user)
        return torch.cat([h(user) for h in self.heads], dim=-1)  # [B, 4]


class NeuralTrainer:
    """Generic trainer with fp16, DataParallel, and early stopping."""

    def __init__(self, model, train_loader, val_loader, class_weights,
                 encoder_lr=1e-5, other_lr=1e-3, patience=3, use_fp16=True):
        # Wrap with DataParallel if 2+ GPUs
        self.raw_model = model
        if NUM_GPUS > 1:
            model = nn.DataParallel(model)
            print(f'  🔀 DataParallel on {NUM_GPUS} GPUs')
        self.model        = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.patience     = patience
        self.scaler       = GradScaler() if use_fp16 and DEVICE.type == 'cuda' else None
        self.use_fp16     = use_fp16 and DEVICE.type == 'cuda'

        # Differential LR
        enc_params = list(self.raw_model.encoder.parameters())
        enc_ids    = set(id(p) for p in enc_params)
        other_p    = [p for p in self.raw_model.parameters() if id(p) not in enc_ids]
        self.opt   = torch.optim.AdamW([
            {'params': enc_params, 'lr': encoder_lr},
            {'params': other_p,    'lr': other_lr},
        ], weight_decay=1e-2)

        # Per-axis BCEWithLogitsLoss with pos_weight
        self.criteria = []
        for cw in class_weights:
            pw = (cw[1]/cw[0]).to(DEVICE) if len(cw) == 2 else torch.tensor(1.0).to(DEVICE)
            self.criteria.append(nn.BCEWithLogitsLoss(pos_weight=pw))

    def _loss(self, logits, labels):
        return sum(c(logits[:, i], labels[:, i])
                   for i, c in enumerate(self.criteria)) / len(self.criteria)

    def train_epoch(self):
        self.model.train()
        total = 0
        for batch in tqdm(self.train_loader, leave=False, desc='Train'):
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            lbls  = batch['labels'].to(DEVICE)
            self.opt.zero_grad()
            if self.use_fp16:
                with autocast():
                    logits = self.model(ids, masks)
                    loss   = self._loss(logits, lbls)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.opt)
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.opt)
                self.scaler.update()
            else:
                logits = self.model(ids, masks)
                loss   = self._loss(logits, lbls)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step()
            total += loss.item()
        return total / len(self.train_loader)

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        all_logits, all_lbls = [], []
        for batch in tqdm(loader, leave=False, desc='Eval'):
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            with autocast() if self.use_fp16 else torch.no_grad():
                logits = self.model(ids, masks)
            all_logits.append(logits.cpu().float())
            all_lbls.append(batch['labels'])
        logits_all = torch.cat(all_logits)
        lbls_all   = torch.cat(all_lbls).int().numpy()
        preds_all  = (torch.sigmoid(logits_all) >= 0.5).int().numpy()
        scores = {n: f1_score(lbls_all[:, i], preds_all[:, i], average='macro', zero_division=0)
                  for i, n in enumerate(DIM_NAMES)}
        scores['Average'] = np.mean(list(scores.values()))
        return lbls_all, preds_all, scores

    def train(self, max_epochs=CONFIG['MAX_EPOCHS']):
        best_f1, patience_cnt, best_state = 0, 0, None
        for ep in range(1, max_epochs+1):
            loss = self.train_epoch()
            _, _, val_sc = self.evaluate(self.val_loader)
            print(f'  Ep{ep:02d} loss={loss:.4f} val_avg={val_sc["Average"]:.4f}')
            if val_sc['Average'] > best_f1:
                best_f1, patience_cnt = val_sc['Average'], 0
                best_state = {k: v.clone() for k, v in self.raw_model.state_dict().items()}
            else:
                patience_cnt += 1
                if patience_cnt >= self.patience:
                    print(f'  ⏹ Early stop at ep{ep} (best={best_f1:.4f})')
                    break
            torch.cuda.empty_cache()
        if best_state:
            self.raw_model.load_state_dict(best_state)


# ── Train RoBERTa ──
print('\n⏳ [RoBERTa-mean] Training...')
roberta_model   = RoBERTaMeanModel()
roberta_trainer = NeuralTrainer(
    roberta_model, train_loader, val_loader, class_weights_list,
    encoder_lr=CONFIG['ENCODER_LR'], other_lr=CONFIG['OTHER_LR'],
    patience=CONFIG['PATIENCE']
)
roberta_trainer.train()

rob_true, rob_preds, rob_scores = roberta_trainer.evaluate(test_loader)

# ── Save RoBERTa predictions ──
rob_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    rob_save[f'y_true_{c}'] = rob_true[:, i]
    rob_save[f'y_pred_{c}'] = rob_preds[:, i]
rob_save.to_csv(f"{CONFIG['RESULT_DIR']}/roberta_predictions.csv", index=False)
print(f'✅ RoBERTa scores: {rob_scores}')
print(f'   Saved → {CONFIG["RESULT_DIR"]}/roberta_predictions.csv')

## 🕸️ Cell 15 — Baseline 3: D-DGCN (Dynamic Deep Graph CNN)

In [ ]:
if not HAS_PYG:
    print('⚠️  Skipping D-DGCN (torch_geometric not available)')
    ddgcn_scores = {n: 0.0 for n in DIM_NAMES + ['Average']}
    ddgcn_true   = np.zeros((len(test_df), 4))
    ddgcn_preds  = np.zeros((len(test_df), 4))
else:
    class DDGCNModel(nn.Module):
        """
        Tensor flow:
        [B,P=50,L]
          → RoBERTa CLS : [B,P,768]
          → Project     : [B,P,256]
          → Per-sample cosine-sim top-K graph → PyG Batch
          → GCNConv(256→128) → ReLU
          → GCNConv(128→64)  → ReLU
          → global_mean_pool : [B,64]
          → 4 heads          : [B,4]
        """
        def __init__(self, model_name=CONFIG['ROBERTA_NAME'],
                     proj_dim=256, gcn_h=CONFIG['GCN_HIDDEN'],
                     gcn_out=CONFIG['GCN_OUT'], top_k=CONFIG['TOP_K_GRAPH'],
                     dropout=0.1):
            super().__init__()
            self.top_k   = top_k
            self.encoder = AutoModel.from_pretrained(model_name)
            self.encoder.gradient_checkpointing_enable()
            roberta_h = self.encoder.config.hidden_size
            self.proj = nn.Linear(roberta_h, proj_dim)
            self.drop = nn.Dropout(dropout)
            self.gcn1 = GCNConv(proj_dim, gcn_h)
            self.gcn2 = GCNConv(gcn_h,    gcn_out)
            self.heads = nn.ModuleList([nn.Linear(gcn_out, 1) for _ in range(4)])

        def _build_graph(self, feats):
            """feats: [P, D] → edge_index: [2, edges]"""
            norm  = F.normalize(feats, dim=-1)
            sim   = torch.mm(norm, norm.t())           # [P, P]
            P     = sim.shape[0]
            k     = min(self.top_k, P-1)
            sim.fill_diagonal_(torch.finfo(sim.dtype).min)
            _, topk_idx = sim.topk(k, dim=-1)          # [P, k]
            src = torch.arange(P, device=feats.device).unsqueeze(1).expand(-1, k).reshape(-1)
            dst = topk_idx.reshape(-1)
            edge_index = torch.stack([
                torch.cat([src, dst]),
                torch.cat([dst, src])
            ], dim=0)
            return edge_index

        def forward(self, input_ids, attention_mask, **kwargs):
            B, P, L = input_ids.shape
            ids   = input_ids.view(B*P, L)
            masks = attention_mask.view(B*P, L)
            out   = self.encoder(input_ids=ids, attention_mask=masks)
            cls   = out.last_hidden_state[:, 0, :]      # [B*P, 768]
            feats = self.drop(F.relu(self.proj(cls)))   # [B*P, 256]
            feats = feats.view(B, P, -1)                # [B, P, 256]

            data_list = []
            for b in range(B):
                ei = self._build_graph(feats[b])
                data_list.append(Data(x=feats[b], edge_index=ei))
            pg = Batch.from_data_list(data_list)

            x  = F.relu(self.gcn1(pg.x, pg.edge_index))  # [B*P, 128]
            x  = self.drop(x)
            x  = F.relu(self.gcn2(x,    pg.edge_index))  # [B*P, 64]
            ue = global_mean_pool(x, pg.batch)            # [B, 64]
            ue = self.drop(ue)
            return torch.cat([h(ue) for h in self.heads], dim=-1)  # [B, 4]

    print('\n⏳ [D-DGCN] Training...')
    ddgcn_model   = DDGCNModel()
    ddgcn_trainer = NeuralTrainer(
        ddgcn_model, train_loader, val_loader, class_weights_list,
        encoder_lr=CONFIG['ENCODER_LR'], other_lr=CONFIG['OTHER_LR'],
        patience=CONFIG['PATIENCE']
    )
    ddgcn_trainer.train()

    ddgcn_true, ddgcn_preds, ddgcn_scores = ddgcn_trainer.evaluate(test_loader)

    # ── Save D-DGCN predictions ──
    dgcn_save = pd.DataFrame()
    for i, c in enumerate(LABEL_COLS):
        dgcn_save[f'y_true_{c}'] = ddgcn_true[:, i]
        dgcn_save[f'y_pred_{c}'] = ddgcn_preds[:, i]
    dgcn_save.to_csv(f"{CONFIG['RESULT_DIR']}/ddgcn_predictions.csv", index=False)
    print(f'✅ D-DGCN scores: {ddgcn_scores}')
    print(f'   Saved → {CONFIG["RESULT_DIR"]}/ddgcn_predictions.csv')

## 📊 Cell 16 — Final Comparison Table

In [ ]:
def build_comparison_table(result_dir=CONFIG['RESULT_DIR']):
    """
    Read all 4 prediction CSVs and compute per-axis + average Macro-F1.
    """
    files = {
        'SVM + TF-IDF':     'svm_predictions.csv',
        'RoBERTa-mean':     'roberta_predictions.csv',
        'D-DGCN':           'ddgcn_predictions.csv',
        'RAG + Multi-Agent':'rag_multi_agent_predictions.csv',
    }
    rows = []
    for name, fname in files.items():
        fpath = os.path.join(result_dir, fname)
        if not os.path.exists(fpath):
            print(f'  ⚠️  {fname} not found, skipping.')
            continue
        df_r = pd.read_csv(fpath)
        row  = {'Model': name}
        f1s  = []
        for col, dim in zip(LABEL_COLS, DIM_NAMES):
            f1 = f1_score(df_r[f'y_true_{col}'], df_r[f'y_pred_{col}'],
                          average='macro', zero_division=0)
            row[f'F1 {dim}'] = round(f1, 4)
            f1s.append(f1)
        row['Avg Macro-F1'] = round(np.mean(f1s), 4)
        rows.append(row)

    return pd.DataFrame(rows).set_index('Model')


print('\n' + '='*65)
print('📊  FINAL COMPARISON — Macro-F1 on Test Set')
print('='*65)
cmp_df = build_comparison_table()
print(cmp_df.to_string())
print('='*65)
print(f'\n✅ All prediction CSVs saved in: {CONFIG["RESULT_DIR"]}')
print('Files:', os.listdir(CONFIG['RESULT_DIR']))